# Notebook 02 — Baselines and Leakage-Safe Validation

**Goal:** Establish honest baselines with `GroupKFold` cross-validation.
Demonstrate the leakage caused by random splitting.
Experiment with RUL clipping at 125 cycles.

---

## Key concept: why we split by engine unit

All rows from the same engine are highly correlated — they share the same
physical unit, same fault mode, same degradation trajectory. A random split
would put rows from engine 5, cycle 50 in training and engine 5, cycle 55 in
validation. The model has already "seen" that engine and its predictions are
trivially good. This is **data leakage** and inflates apparent performance.

The correct approach is `GroupKFold(groups=unit_ids)`: every fold guarantees
that no engine appears in both train and validation.

In [ ]:
import sys; sys.path.insert(0, '..')

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
import seaborn as sns
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

from src.data import load_dataset, save_processed, load_processed, get_last_cycle
from src.features import build_features, get_feature_columns, save_feature_names
from src.evaluate import rmse, mae, nasa_score, compute_all_metrics, evaluate_cv

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

---
## 1. Load and engineer features

In [ ]:
# Load raw data with RUL labels
ds = load_dataset('FD001', clip_rul=None)  # unclipped first
train_raw = ds['train']

print(f'Training set: {train_raw.shape}')
print(f'RUL range: {train_raw["rul"].min()} – {train_raw["rul"].max()}')

In [ ]:
# Build feature matrices (unclipped and clipped)
print('Building features — unclipped …')
train_feat_unclipped = build_features(train_raw, dataset='FD001',
                                       processed_dir=Path('../data/processed'))

from src.data import attach_rul_labels, load_raw
train_clipped = attach_rul_labels(load_raw('FD001', 'train'), clip=125)
print('Building features — clipped at 125 …')
train_feat_clipped = build_features(train_clipped, dataset='FD001',
                                     processed_dir=Path('../data/processed'))

feature_cols = get_feature_columns(train_feat_unclipped)
save_feature_names(feature_cols, 'FD001', Path('../data/processed'))

print(f'Feature columns: {len(feature_cols)}')
print(f'Sample features: {feature_cols[:5]}')

In [ ]:
# Save processed features for downstream notebooks
save_processed(train_feat_unclipped, 'train_FD001_unclipped', Path('../data/processed'))
save_processed(train_feat_clipped,   'train_FD001_clipped',   Path('../data/processed'))

# Also save test set features
test_raw = ds['test']
train_feat_for_test = build_features(train_raw, dataset='FD001',
                                      processed_dir=Path('../data/processed'))  # fit on train
test_feat = build_features(test_raw, dataset='FD001',
                            processed_dir=Path('../data/processed'))
save_processed(test_feat, 'test_FD001', Path('../data/processed'))

print('Processed files saved.')

---
## 2. Baseline 1 — Constant mean predictor

In [ ]:
def constant_baseline_cv(df_feat, feature_cols, clip_label='unclipped', n_splits=5):
    """Evaluate constant-mean predictor with GroupKFold."""
    X = df_feat[feature_cols].values
    y = df_feat['rul'].values.astype(float)
    groups = df_feat['unit'].values

    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
        y_tr, y_val = y[tr_idx], y[val_idx]
        pred = np.full(len(y_val), y_tr.mean())
        m = compute_all_metrics(y_val, pred)
        m['fold'] = fold
        fold_metrics.append(m)

    cv = evaluate_cv(fold_metrics)
    cv['clip'] = clip_label
    cv['model'] = 'Constant mean'
    return cv, fold_metrics

cv_const_unc, _ = constant_baseline_cv(train_feat_unclipped, feature_cols, 'unclipped')
cv_const_clip, _ = constant_baseline_cv(train_feat_clipped,   feature_cols, 'clipped')

print('Constant baseline — unclipped:')
print(f'  RMSE {cv_const_unc["rmse_mean"]:.1f} ± {cv_const_unc["rmse_std"]:.1f}')
print(f'  MAE  {cv_const_unc["mae_mean"]:.1f}  NASA {cv_const_unc["nasa_score_mean"]:.0f}')
print()
print('Constant baseline — clipped at 125:')
print(f'  RMSE {cv_const_clip["rmse_mean"]:.1f} ± {cv_const_clip["rmse_std"]:.1f}')
print(f'  MAE  {cv_const_clip["mae_mean"]:.1f}  NASA {cv_const_clip["nasa_score_mean"]:.0f}')

---
## 3. Baseline 2 — Ridge regression

In [ ]:
def ridge_cv(df_feat, feature_cols, clip_label='unclipped', n_splits=5, alpha=1.0):
    """Ridge regression with GroupKFold."""
    X = df_feat[feature_cols].values
    y = df_feat['rul'].values.astype(float)
    groups = df_feat['unit'].values

    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
        pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
        pipe.fit(X[tr_idx], y[tr_idx])
        pred = np.clip(pipe.predict(X[val_idx]), 0, None)
        m = compute_all_metrics(y[val_idx], pred)
        m['fold'] = fold
        fold_metrics.append(m)

    cv = evaluate_cv(fold_metrics)
    cv['clip'] = clip_label
    cv['model'] = 'Ridge'
    return cv, fold_metrics

cv_ridge_unc,  _ = ridge_cv(train_feat_unclipped, feature_cols, 'unclipped')
cv_ridge_clip, _ = ridge_cv(train_feat_clipped,   feature_cols, 'clipped')

print('Ridge — unclipped:')
print(f'  RMSE {cv_ridge_unc["rmse_mean"]:.1f} ± {cv_ridge_unc["rmse_std"]:.1f}')
print(f'  MAE  {cv_ridge_unc["mae_mean"]:.1f}  NASA {cv_ridge_unc["nasa_score_mean"]:.0f}')
print()
print('Ridge — clipped at 125:')
print(f'  RMSE {cv_ridge_clip["rmse_mean"]:.1f} ± {cv_ridge_clip["rmse_std"]:.1f}')
print(f'  MAE  {cv_ridge_clip["mae_mean"]:.1f}  NASA {cv_ridge_clip["nasa_score_mean"]:.0f}')

---
## 4. Demonstrating leakage — random vs grouped split

This cell deliberately runs a **random** row-split (the wrong way) and
compares the result to grouped CV. The inflated score from random splitting
explains why group-aware validation is mandatory here.

In [ ]:
X = train_feat_clipped[feature_cols].values
y = train_feat_clipped['rul'].values.astype(float)
groups = train_feat_clipped['unit'].values

# ── Random split (WRONG — shown for illustration only) ──────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
random_rmse_list = []
for tr_idx, val_idx in kf.split(X):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])
    pipe.fit(X[tr_idx], y[tr_idx])
    pred = np.clip(pipe.predict(X[val_idx]), 0, None)
    random_rmse_list.append(float(np.sqrt(mean_squared_error(y[val_idx], pred))))

random_rmse_mean = float(np.mean(random_rmse_list))
grouped_rmse_mean = cv_ridge_clip['rmse_mean']

print('Validation strategy comparison (Ridge + clipped RUL):')
print(f'  Random KFold  RMSE: {random_rmse_mean:.1f}   ← inflated by leakage')
print(f'  GroupKFold    RMSE: {grouped_rmse_mean:.1f}   ← honest')
print(f'  Apparent gain from leakage: {grouped_rmse_mean - random_rmse_mean:.1f} RMSE points')
print()
print('The random split looks much better because the model has seen'
      ' other rows from the same engines during training.')

---
## 5. RUL clipping experiment — why it matters

Clipping the RUL target at 125 removes the flat plateau at the start of
engine life where degradation is invisible. Without clipping, the model
wastes capacity fitting those meaningless early rows.

Effect on NASA score: clipping typically **reduces** it because the model
stops over-predicting RUL for engines that are actually near the end.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RUL distribution
axes[0].hist(train_feat_unclipped['rul'], bins=40, alpha=0.6,
             color='steelblue', label='Unclipped', edgecolor='white')
axes[0].hist(train_feat_clipped['rul'],   bins=40, alpha=0.6,
             color='tomato',    label='Clipped @ 125', edgecolor='white')
axes[0].axvline(125, color='black', ls='--', lw=1.5)
axes[0].set_xlabel('RUL (cycles)')
axes[0].set_ylabel('Count')
axes[0].set_title('RUL distribution: unclipped vs clipped')
axes[0].legend()

# RMSE comparison bar chart
models = ['Constant', 'Ridge']
rmse_unc  = [cv_const_unc['rmse_mean'],  cv_ridge_unc['rmse_mean']]
rmse_clip = [cv_const_clip['rmse_mean'], cv_ridge_clip['rmse_mean']]
xpos = np.arange(len(models))
w = 0.35
axes[1].bar(xpos - w/2, rmse_unc,  w, label='Unclipped', color='steelblue', alpha=0.8)
axes[1].bar(xpos + w/2, rmse_clip, w, label='Clipped',   color='tomato',    alpha=0.8)
axes[1].set_xticks(xpos)
axes[1].set_xticklabels(models)
axes[1].set_ylabel('CV RMSE')
axes[1].set_title('GroupKFold CV RMSE: clipped vs unclipped')
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_baseline_comparison.png')

---
## 6. Log results to MLflow

In [ ]:
mlflow.set_tracking_uri('file://' + str(Path('../mlruns').resolve()))
mlflow.set_experiment('predictive-maintenance')

for cv_result, run_label in [
    (cv_const_unc,  'constant_unclipped'),
    (cv_const_clip, 'constant_clipped'),
    (cv_ridge_unc,  'ridge_unclipped'),
    (cv_ridge_clip, 'ridge_clipped'),
]:
    with mlflow.start_run(run_name=run_label):
        mlflow.log_metrics({
            'cv_rmse_mean':       cv_result['rmse_mean'],
            'cv_rmse_std':        cv_result['rmse_std'],
            'cv_mae_mean':        cv_result['mae_mean'],
            'cv_nasa_score_mean': cv_result['nasa_score_mean'],
        })
        mlflow.log_param('clipping', cv_result.get('clip'))
        mlflow.log_param('model',    cv_result.get('model'))

print('All baseline runs logged to MLflow.')

---
## 7. Summary table

In [ ]:
rows = []
for label, cv in [
    ('Constant (unclipped)', cv_const_unc),
    ('Constant (clipped)',   cv_const_clip),
    ('Ridge (unclipped)',    cv_ridge_unc),
    ('Ridge (clipped)',      cv_ridge_clip),
]:
    rows.append({
        'Model': label,
        'RMSE (mean)': f"{cv['rmse_mean']:.1f}",
        'RMSE (std)':  f"± {cv['rmse_std']:.1f}",
        'MAE':         f"{cv['mae_mean']:.1f}",
        'NASA score':  f"{cv['nasa_score_mean']:.0f}",
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
print()
print('Note: Random KFold RMSE (leakage demo):', round(random_rmse_mean, 1),
      '  vs GroupKFold:', round(grouped_rmse_mean, 1))